# Text Simplification

In this lab, we are going to simply experiment with a few text generation models to discover associated APIs.

## Importing the dependencies

First, we are going to import all the dependencies that we will need for this lab. If you cannot run the following code cell, do not forget to [create an environment](https://docs.astral.sh/uv/guides/projects/), to install the dependencies inside of it (using the command `uv add -r requirements.txt`) and to use it as your Jupyter kernel.

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ['HF_HOME'] = os.getcwd() + "/cache/"
from pprint import pprint

import torch
from datasets import load_dataset
from evaluate import load
from transformers import set_seed, pipeline, AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
import numpy as np
import matplotlib.pyplot as plt

set_seed(12345)

## Identifying the best device to run the model

Since we are going to perform a computing-intensive task, we must identify the most efficient device available to perform it. We do so using PyTorch, which is the back-end that we will use in this lab. We prioritize NVIDIA GPUs with CUDA installed, then Apple Silicon GPUs, and finally CPUs if none of the above is found.

If you need help installing the relevant version of PyTorch: https://pytorch.org/get-started/locally/

If you have a NVIDIA GPU but you don't know whether you have CUDA installed or not, type the following command:

```bash
nvcc --version
```

If you have it installed, you should see the CUDA version installed on your computer. Otherwise, you should install a PyTorch-compatible version (as listed [here](https://pytorch.org/get-started/locally/), row "Stable CUDA").

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device('cpu')

print(device)

## Loading the WikiLarge dataset

To train our model, we are going to need the WikiLarge dataset for text simplification. The dataset was produced for the following paper:

> Xingxing Zhang and Mirella Lapata. 2017. Sentence Simplification with Deep Reinforcement Learning. In Proceedings of the 2017 Conference on Empirical Methods in Natural Language Processing, Copenhagen, Denmark. Association for Computational Linguistics.

In [ ]:
dst = load_dataset("bogdancazan/wikilarge-text-simplification", revision="refs/convert/parquet")

In [ ]:
dst["train"][0]

In [ ]:
sari_metric = load("sari")

## Loading the model

In [ ]:
model_checkpoint = "facebook/bart-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint).to(device)

### Setting the hyperparameters

In [ ]:
batch_size = 16
max_input_length = 1024
max_target_length = 128
learning_rate = 2e-5
num_epochs = 1  # For the sake of training speed. In real conditions, 5 would probably be more relevant
warmup_ratio = 0.1
weight_decay = 0.01

### Tokenizing the data

In [ ]:
def preprocess_data(examples):
    model_inputs = tokenizer(examples["Normal"], max_length=max_input_length, truncation=True)

    labels = tokenizer(text_target=examples["Simple"], max_length=max_target_length, truncation=True)

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [ ]:
encoded_datasets = dst.map(preprocess_data, batched=True)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

### Defining the training arguments

In [ ]:
args = Seq2SeqTrainingArguments(
    f"./cache/{model_checkpoint.split("/")[-1]}",
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    load_best_model_at_end=True,
    learning_rate=learning_rate,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    save_safetensors=True,
    weight_decay=weight_decay,
    save_total_limit=3,
    num_train_epochs=num_epochs,
    predict_with_generate=True,
    include_for_metrics=["inputs"],
    fp16=True,
    metric_for_best_model="sari",
    report_to="none"
)

### Setup the Trainer and evaluation function

In [ ]:
def compute_metrics(eval_pred_inputs):
    predictions, labels, inputs = eval_pred_inputs
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_labels_as_lists = [[decoded_label] for decoded_label in decoded_labels]

    inputs = np.where(inputs != -100, inputs, tokenizer.pad_token_id)
    decoded_inputs = tokenizer.batch_decode(inputs, skip_special_tokens=True)

    result = sari_metric.compute(sources=decoded_inputs, predictions=decoded_preds, references=decoded_labels_as_lists)

    return {k: round(v, 4) for k, v in result.items()}

In [ ]:
trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=encoded_datasets["train"],
    eval_dataset=encoded_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)


Now we are good to fine-tune our model on the dataset!

In [ ]:
trainer.train()

## Assessing the model

### Results on validation and test sets

In [ ]:
print("With validation set:")
validation_results_wikilarge = trainer.evaluate()
pprint(validation_results_wikilarge)
print("With test set:")
test_results_wikilarge = trainer.evaluate(eval_dataset=encoded_datasets["test"], metric_key_prefix='test')
pprint(test_results_wikilarge)


### Training and evaluation loss

In [ ]:
log_history = trainer.state.log_history
x = sorted(list({log["step"] for log in log_history}))
y1 = [log["loss"] if "loss" in log else log["train_loss"] for log in list(filter(lambda log: ("loss" in log) or ("train_loss" in log), log_history))]
y2 = [log["eval_loss"] for log in list(filter(lambda log: "eval_loss" in log, log_history))]

if len(x) < len(y1) or len(x) < len(y2):
    y1 = y1[:len(x)]
    y2 = y2[:len(x)]

fig, ax = plt.subplots()
ax.plot(x, y1, 'r', label="train_loss")
ax.plot(x, y2, 'g', label="eval_loss")
ax.set_xlabel("Step", fontsize='large')
ax.set_ylabel("Loss", fontsize='large')
ax.legend()
plt.tight_layout()

## Experimenting with our newly trained model

In [ ]:
model_trained = trainer.model
simplification_pipeline = pipeline("text2text-generation", model=model_trained, max_length=max_target_length, tokenizer=tokenizer, device=device)

In [ ]:
text = "It was the best of times, it was the worst of times, and while people thrived, others suffered."
simplification_pipeline(text)[0]["generated_text"]

In [ ]:
trainer.save_model("./simplification")